# 📱 Mobile App Data Analysis: What Makes an App Popular?

---

## Introduction

> Our goal: understand what drives app popularity so we can build smarter, not just more.

- This project analyzes data from mobile apps to help our development team make better decisions about what to build next. Since our apps are free and our revenue comes from in-app ads, **the number of users directly determines how much we earn**.
- The goal of this project is to identify what types of apps are most likely to attract a large number of users. By understanding patterns in successful apps, we can guide our developers toward building apps with the highest potential audience on **Google Play** and the **App Store**.

---

In [31]:
import csv
from csv import  reader
apple_store_dataset = list(reader(open('AppleStore.csv',  encoding='utf-8')))
google_play_dataset = list(reader(open('googleplaystore.csv',  encoding='utf-8')))
print(apple_store_dataset[0])
print()
print(google_play_dataset[0])

['id', 'track_name', 'size_bytes', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'user_rating', 'user_rating_ver', 'ver', 'cont_rating', 'prime_genre', 'sup_devices.num', 'ipadSc_urls.num', 'lang.num', 'vpp_lic']

['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']


In [32]:
def explore_data(dataset, start, end, rows_and_columns=False):
    dataset_slice = dataset[start:end]    
    for row in dataset_slice:
        print(row)
        print('\n')
    if rows_and_columns:
        print('Number of rows:', len(dataset))
        print('Number of columns:', len(dataset[0]))

# Headers
apple_store_header = apple_store_dataset[0]
google_play_header = google_play_dataset[0]

# Apple Store
print(apple_store_header)
print()
explore_data(apple_store_dataset, 1, 2, True)

print('\n\n\n')

# Google Play
print(google_play_header)
print()
explore_data(google_play_dataset, 1, 2, True)


['id', 'track_name', 'size_bytes', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'user_rating', 'user_rating_ver', 'ver', 'cont_rating', 'prime_genre', 'sup_devices.num', 'ipadSc_urls.num', 'lang.num', 'vpp_lic']

['284882215', 'Facebook', '389879808', 'USD', '0.0', '2974676', '212', '3.5', '3.5', '95.0', '4+', 'Social Networking', '37', '1', '29', '1']


Number of rows: 7198
Number of columns: 16




['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']

['Photo Editor & Candy Camera & Grid & ScrapBook', 'ART_AND_DESIGN', '4.1', '159', '19M', '10,000+', 'Free', '0', 'Everyone', 'Art & Design', 'January 7, 2018', '1.0.0', '4.0.3 and up']


Number of rows: 10842
Number of columns: 13


## Relevant Columns

> We rely on a small set of columns from each dataset to measure popularity and identify what drives it.

| Purpose | Apple Store Columns | Google Play Columns |
|---|---|---|
| **Popularity metric** | `rating_count_tot` | `Installs` |
| **App label (identifier only)** | `track_name` | `App` |
| **Attractiveness factors** | `user_rating`, `prime_genre`, `cont_rating`, `price`, `lang.num`, `sup_devices.num` | `Rating`, `Category`, `Content Rating`, `Type` |

---

### Dataset Documentation
- [AppleStore](https://www.kaggle.com/datasets/ramamet4/app-store-apple-data-set-10k-apps)
- [googleplaystore](https://www.kaggle.com/datasets/lava18/google-play-store-apps)

# 
***Deleting Incomplete row***
* In the code below we deleted a row **whose length is not compatible with the length of the header**: as it means this row has missing values( though we are assuming relative position of row's elements is not swapped for simplicity)

---

***Checking for Duplicate Apps***
* We searched for duplicates of apps. We aslo gave several examples for duplicated apps' names and duplicated rows whose app name is same.


In [33]:
# Checking for Incomplete row
google_play_dataset = [row  for row in google_play_dataset  if len(row) == len(google_play_header)]
print(len(google_play_dataset))     

# Checking for Duplicate Apps
duplicate_apps = []
unique_apps = []
for row in google_play_dataset[1:]:
    app_name = row[0]
    if app_name in unique_apps:
        duplicate_apps.append(app_name)
    else:
        unique_apps.append(app_name)
# examples of duplicated apps
print(duplicate_apps[:3])  
print()

# example of such duplicated row
for row in google_play_dataset:
    if row[0] == 'Box':
        print(row)
print()
print('Number of duplicate apps:', len(duplicate_apps))


10841
['Quick PDF Scanner + OCR FREE', 'Box', 'Google My Business']

['Box', 'BUSINESS', '4.2', '159872', 'Varies with device', '10,000,000+', 'Free', '0', 'Everyone', 'Business', 'July 31, 2018', 'Varies with device', 'Varies with device']
['Box', 'BUSINESS', '4.2', '159872', 'Varies with device', '10,000,000+', 'Free', '0', 'Everyone', 'Business', 'July 31, 2018', 'Varies with device', 'Varies with device']
['Box', 'BUSINESS', '4.2', '159872', 'Varies with device', '10,000,000+', 'Free', '0', 'Everyone', 'Business', 'July 31, 2018', 'Varies with device', 'Varies with device']

Number of duplicate apps: 1181


## Removing Duplicates — Our Criterion

We will not remove duplicates randomly. Instead, we will keep the row with the **highest number of reviews** for each duplicate app.

> **Why?** The number of reviews increases over time, so the row with the most reviews is the most **recent and up-to-date** entry.

**Approach:**

- We built a dictionary `reviews_max` where each key is an app name and each value is the highest review count found for that app.
- We then looped through the dataset and kept only the row with the highest review count for each app, storing the results in `google_play_clean`.
- We used `already_added` to make sure each app is added to `google_play_clean` only once, even if two rows share the same highest review count.

---


In [34]:
# Removing Duplicate Rows
#key of reviews_max dict is app's name and value is reviews of that app
reviews_max = {}
for row in google_play_dataset[1:]:
    app_name = row[0]
    n_reviews = float(row[3])
    if app_name in reviews_max and n_reviews > reviews_max[app_name]:
        reviews_max[app_name] = n_reviews
    elif app_name not in reviews_max:
        reviews_max[app_name] = n_reviews
google_play_clean = []
already_added = []
for row in google_play_dataset[1:]:
    app_name = row[0]
    n_reviews = float(row[3])
    if n_reviews == reviews_max[app_name] and app_name not in already_added:
        google_play_clean.append(row)
        already_added.append(app_name)
print(len(google_play_clean))

    


9659


In the code below, we  did the following:

  ***Filtering Non-English Apps***

We detect English apps by checking for characters outside the standard ASCII range. Apps with more than 3 such unusual characters are excluded.

- Applied the filter to **Google Play** data → `google_clean`
- Applied the filter to **App Store** data → `apple_clean`

---

  ***Isolating Free Apps***

Since our company only builds free, ad-supported apps, we filter both datasets to keep only free apps (header row retained for reference).

- **Google Play:** kept rows where `Type == 'Free'` → `free_clean_google`
- **App Store:** kept rows where `price == 0.0` → `free_clean_apple` 

In [35]:
# Filtering Non-English Apps

# function to determine  english apps
def detector(string):
    unusual_letters = []
    for letter in string:
        if len(unusual_letters) > 3:
            return False
        elif ord(letter) >= 127:
            unusual_letters.append(letter)
            
    return 'True'

# google play 
google_clean = []
for row in google_play_clean:
    if detector(row[0]) == 'True':
        google_clean.append(row)

# apple store
apple_clean = []
for row in apple_store_dataset:
    if detector(row[0]) == 'True':
        apple_clean.append(row)


# Isolating Free Apps 

# Google play
free_clean_google = []
free_clean_google.append(google_play_dataset[0])
for row in google_clean[1:]:
    if row[6] == 'Free':
        free_clean_google.append(row)


# Apple store
free_clean_apple = []
free_clean_apple.append(apple_store_dataset[0])
for row in apple_clean[1:]:
    if float(row[4]) == 0.0:
        free_clean_apple.append(row)
print(free_clean_apple[:5])
print(free_clean_google[:5])
            

[['id', 'track_name', 'size_bytes', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'user_rating', 'user_rating_ver', 'ver', 'cont_rating', 'prime_genre', 'sup_devices.num', 'ipadSc_urls.num', 'lang.num', 'vpp_lic'], ['284882215', 'Facebook', '389879808', 'USD', '0.0', '2974676', '212', '3.5', '3.5', '95.0', '4+', 'Social Networking', '37', '1', '29', '1'], ['389801252', 'Instagram', '113954816', 'USD', '0.0', '2161558', '1289', '4.5', '4.0', '10.23', '12+', 'Photo & Video', '37', '0', '29', '1'], ['529479190', 'Clash of Clans', '116476928', 'USD', '0.0', '2130805', '579', '4.5', '4.5', '9.24.12', '9+', 'Games', '38', '5', '18', '1'], ['420009108', 'Temple Run', '65921024', 'USD', '0.0', '1724546', '3842', '4.5', '4.0', '1.6.2', '9+', 'Games', '40', '5', '1', '1']]
[['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver'], ['U Launcher Lite – FREE Live Cool Themes, Hide Apps', 'A

## Most Common Apps by Genre

Recall that our end goal is to add the app to both the App Store and Google Play. Our validation strategy for an app idea has three steps:

1. Build a minimal Android version of the app and add it to Google Play.
2. If the app receives a good response from users, develop it further.
3. If the app turns out to be profitable after six months, build an iOS version and add it to the App Store as well.
---
Because our end goal is to add the app on both Google Play and the App Store, we need to find app profiles that are successful on **both** markets. Let's begin the analysis by getting a sense of the most common genres for each market. For this, we'll build frequency tables for a few columns in our datasets.

In [36]:
# gets dataset and index of a column and returns frequency table for the column.
def freq_table(dataset, index):
    frequency_table = {} # key = value in column, frequency_table[key] = frequency of the value in a column 
    for row in dataset[1:]:
        if row[index] not in frequency_table:
            frequency_table[row[index]] = 1
        else: 
            frequency_table[row[index]] += 1
    return frequency_table
    
# gets dataset, index of the column and retunrs its frequecy table in descending order
def display_table(dataset, index):
    table = freq_table(dataset, index)
    table_display = []
    for key in table:
        key_val_as_tuple = (table[key], key)
        table_display.append(key_val_as_tuple)

    table_sorted = sorted(table_display, reverse = True)
    table_dict = {}
    for entry in table_sorted:
        table_dict[entry[1]] = entry[0]
    return table_dict
        
# Using display function for 'prime_genre', 'Genres'and 'Category'
print("Frequency table for 'prime genre' column in apple store")
print(display_table(free_clean_apple , 11))
print()
print("Frequency table for 'Category' column in google play")
print(display_table(free_clean_google, 1))
print()
print("Frequency table for 'Genres' column in google play")
print((display_table(free_clean_google, 9)))


Frequency table for 'prime genre' column in apple store
{'Games': 2257, 'Entertainment': 334, 'Photo & Video': 167, 'Social Networking': 143, 'Education': 132, 'Shopping': 121, 'Utilities': 109, 'Lifestyle': 94, 'Finance': 84, 'Sports': 79, 'Health & Fitness': 76, 'Music': 67, 'Book': 66, 'Productivity': 62, 'News': 58, 'Travel': 56, 'Food & Drink': 43, 'Weather': 31, 'Reference': 20, 'Navigation': 20, 'Business': 20, 'Catalogs': 9, 'Medical': 8}

Frequency table for 'Category' column in google play
{'FAMILY': 1675, 'GAME': 862, 'TOOLS': 750, 'BUSINESS': 407, 'LIFESTYLE': 347, 'PRODUCTIVITY': 345, 'FINANCE': 328, 'MEDICAL': 313, 'SPORTS': 301, 'PERSONALIZATION': 294, 'COMMUNICATION': 287, 'HEALTH_AND_FITNESS': 273, 'PHOTOGRAPHY': 261, 'NEWS_AND_MAGAZINES': 248, 'SOCIAL': 236, 'TRAVEL_AND_LOCAL': 207, 'SHOPPING': 199, 'BOOKS_AND_REFERENCE': 191, 'DATING': 165, 'VIDEO_PLAYERS': 159, 'MAPS_AND_NAVIGATION': 124, 'FOOD_AND_DRINK': 110, 'EDUCATION': 103, 'ENTERTAINMENT': 85, 'LIBRARIES_AND_D

## Apps With the Most Users

We calculate the **average number of users per genre/category** for each platform, sorted from highest to lowest.

- **App Store:** grouped by `prime_genre`, averaging `rating_count_tot` (used as a proxy for installs, since exact install counts aren't available)
- **Google Play:** grouped by `Category`, averaging `Installs` (cleaned by removing `+` and `,` characters)

In [37]:
# Apps with the most users

# average users for each genre of 'prime genre'
average_users_prime = []
for key in display_table(free_clean_apple , 11) :
    total = 0 
    len_genre = 0
    for row in free_clean_apple:
        if row[11] == key:
            total += float(row[5])
            len_genre += 1
    average = (total / len_genre)
    
    average_users_prime.append((key, average))
average_users_prime.sort(key=lambda x:x[1], reverse=True)
for tuple in average_users_prime :
    print(tuple)
print()

# average users for each genre of 'Category'
average_users_Category = []
for key in display_table(free_clean_google , 1) :
    total = 0 
    len_genre = 0
    for row in free_clean_google:
        if row[1] == key:
            total += float(row[5].replace('+', '').replace(',', ''))
            len_genre += 1
    average = (total / len_genre)
    
    average_users_Category.append((key, average))
average_users_Category.sort(key=lambda x:x[1], reverse=True)
for tuple in average_users_Category:
    print(tuple)

    
    
    

('Reference', 67447.9)
('Music', 56482.02985074627)
('Social Networking', 53078.195804195806)
('Weather', 47220.93548387097)
('Photo & Video', 27249.892215568863)
('Navigation', 25972.05)
('Travel', 20216.01785714286)
('Food & Drink', 20179.093023255813)
('Sports', 20128.974683544304)
('Health & Fitness', 19952.315789473683)
('Productivity', 19053.887096774193)
('Games', 18924.68896765618)
('Shopping', 18746.677685950413)
('News', 15892.724137931034)
('Utilities', 14010.100917431193)
('Finance', 13522.261904761905)
('Entertainment', 10822.961077844311)
('Lifestyle', 8978.308510638299)
('Book', 8498.333333333334)
('Business', 6367.8)
('Education', 6266.333333333333)
('Catalogs', 1779.5555555555557)
('Medical', 459.75)

('COMMUNICATION', 38456119.167247385)
('VIDEO_PLAYERS', 24727872.452830188)
('SOCIAL', 23253652.127118643)
('PHOTOGRAPHY', 17840110.40229885)
('PRODUCTIVITY', 16787331.344927534)
('GAME', 15588015.603248259)
('TRAVEL_AND_LOCAL', 13984077.710144928)
('ENTERTAINMENT', 11640

# App Profile Recommendation

## Key Observations
Both markets below are led by genres skewed by one or two giant, already-dominant apps:

- **App Store:** Reference (67.4K avg ratings) — Bible, Dictionary.com; Music (56.5K) — Spotify; Social Networking (53.1K) — Facebook, Pinterest.
- **Google Play:** Communication (38.5M avg installs) — WhatsApp, Messenger; Video Players (24.7M); Social (23.3M).

These top spots aren't realistic targets for a new developer — the high averages come from a handful of outliers, not broad category demand.

## Recommendation: Productivity

| Market | Signal |
|---|---|
| Google Play | ~16.8M avg installs — strong and broad-based (calendars, notes, PDF tools, task managers), not monopolized by a single app |
| App Store | Performs solidly, well above Business, Education, Finance — evergreen demand for notes/planners/scanners |

**Why it works better than the "top" genres:**
- Not dominated by 1–2 giants → realistic for a small team to enter
- Present with solid, representative numbers on **both** stores
- Room for differentiation (habit trackers, PDF/OCR scanners, time-blocking apps)

**Suggested app concept:** A minimalist habit/task tracker combined with a simple utility (e.g., quick PDF scan or time-blocking).